# VLA Manipulation Pipeline - Google Colab Verification & Execution Notebook

This notebook runs Phase 0 verification, task simulation, demo collection, SmolVLA fine-tuning, and evaluation.

## 1. Confirm GPU + CUDA

In [ ]:
import torch
print("Torch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA is not available. Please go to Runtime -> Change runtime type -> T4/A100 GPU.")

## 2. Install & Verify LeRobot CLI (with optional dataset/av dependencies)

In [ ]:
!pip install -q "lerobot[dataset]" av
!lerobot-train --help

## 3. Install & Verify ManiSkill3 (Headless EGL Rendering)

In [ ]:
# Install ManiSkill3 without strict pinned mplib dependency
!pip install -q mplib==0.2.0
!pip install -q mani_skill --no-deps
!pip install -q gymnasium torchtriton h5py trimesh transforms3d pandas

import os
import mani_skill.envs
import gymnasium as gym
import matplotlib.pyplot as plt

# Ensure headless EGL rendering configuration for Colab
os.environ["SAPIEN_RENDER_ENGINE"] = "EGL"

# Instantiate environment
env = gym.make("PickCube-v1", obs_mode="rgbd", render_mode="rgb_array")
obs, _ = env.reset()

# Step environment with random action
action = env.action_space.sample()
obs, reward, terminated, truncated, info = env.step(action)

# Render frame to verify visual output
frame = env.render()
if isinstance(frame, list):
    frame = frame[0]

plt.figure(figsize=(6, 6))
plt.imshow(frame)
plt.title("ManiSkill3 PickCube-v1 Verification Frame")
plt.axis("off")
plt.savefig("maniskill_verification_frame.png")
print("Frame successfully rendered and saved to maniskill_verification_frame.png")
env.close()

## 4. Mounting Google Drive for Persistence

In [ ]:
from google.colab import drive
import os

# Mount Google Drive for persistent checkpoints and results storage
drive.mount('/content/drive')

project_dir = "/content/drive/MyDrive/vla-manipulation"
os.makedirs(project_dir, exist_ok=True)
print(f"Project persistence directory ready at: {project_dir}")